# Verifying the implementation of logical Cliffords

Motivation: Logical Cliffords can have very non-obvious implementations, especially for $k>1$ codes. We want to be able to validate implementations and catch implementation bugs in our logical gadgets.

Example: check out this paper on CSD codes from Noah and Elijah (Table 2) -> https://arxiv.org/pdf/2510.18753

## What are we verifying?

* This is a simple tool which checks logical Clifford semantics against physical implementation
* Works for $k>1$ codes and also interblock operations
* This approach is fairly general and works for any stabilizer code.

## Basic example for Steane

In [ ]:
from typing import no_type_check


from guppylang import guppy
from guppylang.std.builtins import array
from guppylang.std.mem import mem_swap
from guppylang.std.quantum import qubit, h, cx, rx
from guppylang.std.qsystem.helios import zz_phase
from guppylang.std.angles import pi

from guppyft.verifier import (
    check_clifford_semantics,
    check_stabilizer_state_semantics
)
from guppyft.code_def import StabilizerCode

In [ ]:
STEANE_DEF = StabilizerCode.from_python_strings(
    num_physical_qubits=7,
    num_logical_qubits=1,
    distance=3,
    generators=["XXXXIII", "IXXIXXI", "IIXXIXX", "ZZZZIII", "IZZIZZI", "IIZZIZZ"],
    x_logicals=["XXXXXXX"],
    z_logicals=["ZZZZZZZ"],
)

In [ ]:

@guppy
@no_type_check
def steane_logical_h(qs: array[qubit, 1]) -> None:
    h(qs[0])


@guppy
@no_type_check
def steane_physical_h(block: array[qubit, 7]) -> None:
    for i in range(len(block)):
        h(block[i])

In [ ]:
check_clifford_semantics(steane_logical_h, steane_physical_h, STEANE_DEF)

In [ ]:
@guppy
@no_type_check
def steane_physical_cx(
    first_block: array[qubit, 7], second_block: array[qubit, 7]
) -> None:
    for i in range(len(first_block)):
        cx(first_block[i], second_block[i])


## Code definition

```python
@dataclass(frozen=True)
class StabilizerCode:
    num_physical_qubits: int
    num_logical_qubits: int
    distance: int
    generators: pauli.StringSet
    x_logicals: pauli.Strings
    z_logicals: pauli.Strings
```

Some `__post_init__` validation to check that the stabilizers commute and other sanity checks.

In [14]:
CSS_4Q_DEF = StabilizerCode.from_python_strings(
    num_physical_qubits=4,
    num_logical_qubits=2,
    distance=2,
    generators=["XXXX", "ZZZZ"],
    x_logicals=["XXII", "XIXI"],
    z_logicals=["IZIZ", "IIZZ"],
)


## Less trivial $[[4, 2, 2]]$ examples

In [15]:
@guppy
@no_type_check
def css_4_2_2_double_h_logical(block: array[qubit, 2]) -> None:
    h(block[0])
    h(block[1])


@guppy
@no_type_check
def css_4_2_2_double_h_physical(block: array[qubit, 4]) -> None:
    h(block[0])
    h(block[1])
    h(block[2])
    h(block[3])
    mem_swap(block[1], block[2])

In [16]:
@guppy
@no_type_check
def css_4_2_2_transversal_cx_logical(
    first_block: array[qubit, 2], second_block: array[qubit, 2]
) -> None:
    for i in range(2):
        cx(first_block[i], second_block[i])


@guppy
@no_type_check
def css_4_2_2_transversal_cx_physical(
    first_block: array[qubit, 4], second_block: array[qubit, 4]
) -> None:
    for i in range(4):
        cx(first_block[i], second_block[i])

In [17]:
@guppy
@no_type_check
def css_4_2_2_non_ft_zero() -> array[qubit, 4]:
    block = array(qubit() for _ in range(4))
    h(block[2])
    cx(block[2], block[1])
    cx(block[2], block[3])
    cx(block[1], block[0])
    return block


In [18]:
@guppy
@no_type_check
def css_4_2_2_intra_block_cx_logical(block: array[qubit, 2]) -> None:
    cx(block[0], block[1])


@guppy
@no_type_check
def css_4_2_2_intra_block_cx_physical(block: array[qubit, 4]) -> None:
    mem_swap(block[3], block[1])

In [19]:
check_clifford_semantics(css_4_2_2_intra_block_cx_logical, css_4_2_2_intra_block_cx_physical, CSS_4Q_DEF)

In [21]:
@guppy
@no_type_check
def css_4_2_2_addressable_rx_minus_half_pi_logical(block: array[qubit, 2]) -> None:
    rx(block[1], -pi / 2)


@guppy
@no_type_check
def css_4_2_2_addressable_rx_minus_half_pi_physical(block: array[qubit, 4]) -> None:
    h(block[0])
    h(block[2])
    zz_phase(block[0], block[2], -pi / 2)
    h(block[0])
    h(block[2])

In [22]:
check_clifford_semantics(
    css_4_2_2_addressable_rx_minus_half_pi_logical,
    css_4_2_2_addressable_rx_minus_half_pi_physical,
    CSS_4Q_DEF
)

## Trying to explain it step by step

Let's go back to the logical Hadamard in Steane

In [ ]:
# Get the 2k stabilizers for the 2k qubit Choi state encoding the logical operation.
semantic_choi_stabilizers = compute_stabilizers_single_block(
    steane_logical_h,
    choi_state_preparation=default_choi_state_preparation,
    n_func_qubits=STEANE.num_logical_qubits,
)

In [ ]:
semantic_choi_stabilizers.to_dataframe()

In [ ]:
# Expand the 2k logical stabilizers to 2k stabilizers of size 2n.
# We also add the 2(n-k) stabilizer generators of our code.
# For each code block there are (n-k) so 2 blocks give us 2(n-k).
# We have 2k + 2(n-k) = 2n stabilizers in total.
expanded_semantic_stabilizers = get_expanded_stabilizer_set(
    semantic_choi_stabilizers, STEANE, num_blocks=1
)

$$
X_L \mapsto XXXXXXX\, \qquad Z_L \mapsto ZZZZZZZ
$$

In [ ]:
expanded_semantic_stabilizers.to_dataframe()

In [ ]:
expanded_semantic_stabilizers.canonicalize_all()  # Normalize Clifford tableau
expanded_semantic_stabilizers.to_dataframe()

In [ ]:
# Calculate the 2n stabilizers of the Choi state encoding the physical operation.
implementation_stabilizers = compute_stabilizers_single_block(
    steane_physical_h,
    steane_choi_state,
    STEANE.num_physical_qubits,
)

In [ ]:
implementation_stabilizers.to_dataframe()

In [ ]:
implementation_stabilizers.canonicalize_all()  # Normalize Clifford tableau
implementation_stabilizers.to_dataframe()

In [ ]:
expanded_semantic_stabilizers == implementation_stabilizers

## Things to add?

* Testing for non-CSS codes. $[[5, 1, 3]], [[4, 2, 2]] (\text{non-CSS variant})$
* Automatically prepare logical Bell states under the hood... Much friendlier and easier to test non-CSS codes
* Allow validating implementations which use ancilla qubits.
* How feasible is it to do non-Cliffords as well?